# 06 — Logical Operators, Code Distance, and Degeneracy

## Read before you begin

- Arthur Pesah, *Stabilizer Formalism II* (https://arthurpesah.me/blog/2023-03-16-stabilizer-formalism-2/).
- Exact sections to read: **Logical qubits**, **Pauli logical operators**, **Logical cosets**.
- Concepts to extract: why $k = n - m$ for $m$ independent stabilizer generators; the centralizer of the stabilizer group and how nontrivial logical operators arise from it; logical $X$ and $Z$ operators and logical equivalence modulo the stabilizer group; code distance defined via logical operator weight.

## Learning objectives

Notebooks 03 and 05 used stabilizers, syndromes, and (in Shor's case) an ad hoc brute-force distance check. This notebook makes those ideas precise and general. By the end you should be able to:

1. explain why $m$ independent stabilizer generators on $n$ qubits leave $k = n - m$ logical qubits, and see the dimension halve with each added generator;
2. define the centralizer $C(S)$ of a stabilizer group $S$, and explain why nontrivial logical operators are exactly the elements of $C(S)$ that are *not* in $S$ itself;
3. build logical $\bar{X}$ and $\bar{Z}$ operators and verify they anticommute, exactly like physical $X$ and $Z$;
4. show that logical operators differing by a stabilizer element are logically equivalent (they act identically on the code space);
5. state the general definition of code distance (minimum weight of a nontrivial logical operator) and apply one algorithm to compute it for the repetition code, Shor's code, and the Steane code.

**Expected outputs:** a demonstration of dimension-halving on the repetition code, an explicit centralizer/coset enumeration for the repetition code, and a single reusable `code_distance` routine whose output matches what notebooks 02, 03, and 05 already established (distance 3 for all three codes examined here) — computed the same way for each, instead of three different ad hoc arguments.

**Assumes:** notebooks 02, 03, and 05 (the repetition, Shor, and Steane codes, reused here as worked examples rather than rebuilt from scratch).

## 1. Setup: Pauli machinery, and the three codes as data

We reuse the exact stabilizer generator strings from earlier notebooks, so this notebook's `code_distance` routine can be tested on genuinely independent, already-verified examples.

In [1]:
import numpy as np
from itertools import combinations, product as iproduct

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
PAULI = {"I": I2, "X": X, "Y": Y, "Z": Z}
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)


def kron_all(items):
    result = items[0]
    for m in items[1:]:
        result = np.kron(result, m)
    return result


def pauli_string_matrix(string):
    return kron_all([PAULI[ch] for ch in string])


def basis_state(bits):
    return kron_all([ket1 if b else ket0 for b in bits])


# Notebook 02/03's 3-qubit repetition code.
repetition_generators = ["ZZI", "IZZ"]
repetition_0L = basis_state((0, 0, 0))
repetition_1L = basis_state((1, 1, 1))

# Notebook 05's Shor code.
shor_generators = ["ZZIIIIIII", "IZZIIIIII", "IIIZZIIII", "IIIIZZIII",
                   "IIIIIIZZI", "IIIIIIIZZ", "XXXXXXIII", "IIIXXXXXX"]
shor_block_plus = basis_state((0, 0, 0)) + basis_state((1, 1, 1))
shor_block_minus = basis_state((0, 0, 0)) - basis_state((1, 1, 1))
shor_0L = kron_all([shor_block_plus] * 3)
shor_0L = shor_0L / np.linalg.norm(shor_0L)
shor_1L = kron_all([shor_block_minus] * 3)
shor_1L = shor_1L / np.linalg.norm(shor_1L)

# Notebook 05's Steane code (built from notebook 04's Hamming H).
def bits_of(j, width):
    return tuple((j >> shift) & 1 for shift in reversed(range(width)))

H_columns = [bits_of(j, width=3) for j in range(1, 8)]
H = tuple(tuple(col[row] for col in H_columns) for row in range(3))
steane_generators = (["".join("X" if b else "I" for b in row) for row in H]
                     + ["".join("Z" if b else "I" for b in row) for row in H])

print("Repetition:", repetition_generators)
print("Shor:      ", shor_generators)
print("Steane:    ", steane_generators)

Repetition: ['ZZI', 'IZZ']
Shor:       ['ZZIIIIIII', 'IZZIIIIII', 'IIIZZIIII', 'IIIIZZIII', 'IIIIIIZZI', 'IIIIIIIZZ', 'XXXXXXIII', 'IIIXXXXXX']
Steane:     ['IIIXXXX', 'IXXIIXX', 'XIXIXIX', 'IIIZZZZ', 'IZZIIZZ', 'ZIZIZIZ']


## 2. Why $k = n - m$: dimension-halving

Each stabilizer generator $S_i$ is a Hermitian, unitary operator with eigenvalues $\pm 1$, so it splits the full $2^n$-dimensional space into two equal halves: a $+1$-eigenspace and a $-1$-eigenspace, each of dimension $2^{n-1}$. Demanding a *second, independent* generator also equal $+1$ cuts whatever space remains in half again — "independent" is doing real work here: it guarantees the new constraint isn't already implied by (or contradictory to) the first, so it genuinely halves the space rather than leaving it unchanged or emptying it. After $m$ independent constraints, the joint $+1$-eigenspace — the code space — has dimension $2^{n-m} = 2^k$.

We can watch this happen one generator at a time on the repetition code (3 qubits, so the full space is only 8-dimensional).

In [2]:
def projector_onto_plus_one(generator_strings, n_qubits):
    dim = 2 ** n_qubits
    P = np.eye(dim, dtype=complex)
    for s in generator_strings:
        S = pauli_string_matrix(s)
        P = P @ ((np.eye(dim, dtype=complex) + S) / 2)
    return P


print("Applying repetition-code generators one at a time:")
print(f"  no constraints:        dimension 2^3 = 8")
for i in range(1, len(repetition_generators) + 1):
    P = projector_onto_plus_one(repetition_generators[:i], n_qubits=3)
    rank = np.linalg.matrix_rank(P, tol=1e-8)
    print(f"  after {i} generator(s): dimension {rank}  (2^(3-{i}) = {2**(3-i)})")

Applying repetition-code generators one at a time:
  no constraints:        dimension 2^3 = 8
  after 1 generator(s): dimension 4  (2^(3-1) = 4)
  after 2 generator(s): dimension 2  (2^(3-2) = 2)


### Prediction 1

**Before running the next cell:** the Steane code has $n=7$ qubits and 6 independent generators (3 $X$-type, 3 $Z$-type, from notebook 05). What dimension do you expect its code space to have, and how many logical qubits $k$ does that correspond to?

In [3]:
for name, gens, n in [("Repetition", repetition_generators, 3),
                       ("Shor", shor_generators, 9),
                       ("Steane", steane_generators, 7)]:
    P = projector_onto_plus_one(gens, n_qubits=n)
    rank = np.linalg.matrix_rank(P, tol=1e-8)
    k = round(np.log2(rank))
    print(f"{name:<10} n={n}, m={len(gens)} generators -> code space dimension {rank} = 2^{k}, so k={k}")

Repetition n=3, m=2 generators -> code space dimension 2 = 2^1, so k=1
Shor       n=9, m=8 generators -> code space dimension 2 = 2^1, so k=1
Steane     n=7, m=6 generators -> code space dimension 2 = 2^1, so k=1


## 3. The centralizer and where logical operators come from

The **centralizer** $C(S)$ of the stabilizer group $S$ is the set of *all* Pauli operators that commute with *every* generator of $S$ (not just the elements of $S$ itself, which trivially commute with each other since $S$ is abelian). Because $S \subseteq C(S)$, the centralizer splits into $S$ itself, plus everything else in $C(S)$.

- An operator in $S$ acts as a **scalar** ($\pm 1$) on every code state — it's a *check*, not a logical operation.
- An operator in $C(S) \setminus S$ (in the centralizer but not the stabilizer group) commutes with every check, so it maps the code space to itself, but acts **nontrivially** there — this is exactly what makes it a **logical operator**: undetectable by any syndrome measurement (since it commutes with every stabilizer, no check ever fires), yet it changes the encoded logical state.

We can see this whole structure explicitly for the repetition code, since its full Pauli group on 3 qubits only has $4^3 = 64$ elements — small enough to enumerate completely.

In [4]:
def commutes(p, q):
    disagreements = sum(1 for a, b in zip(p, q) if a != "I" and b != "I" and a != b)
    return disagreements % 2 == 0


def all_pauli_strings(n):
    return ["".join(letters) for letters in iproduct("IXYZ", repeat=n)]


def weight(pauli_string):
    return sum(1 for ch in pauli_string if ch != "I")


centralizer = [p for p in all_pauli_strings(3) if all(commutes(p, g) for g in repetition_generators)]
print(f"Full Pauli group on 3 qubits has 64 elements; the centralizer C(S) has {len(centralizer)} elements.")
print(sorted(centralizer, key=weight))

Full Pauli group on 3 qubits has 64 elements; the centralizer C(S) has 16 elements.
['III', 'IIZ', 'IZI', 'ZII', 'IZZ', 'ZIZ', 'ZZI', 'XXX', 'XXY', 'XYX', 'XYY', 'YXX', 'YXY', 'YYX', 'YYY', 'ZZZ']


## 4. Classifying centralizer elements: trivial, or a logical operator?

For each element of $C(S)$, look at how it acts on the 2-dimensional code space spanned by $|0_L\rangle, |1_L\rangle$: build the $2\times 2$ "logical action" matrix of overlaps. If that matrix is a multiple of the identity, the operator is trivial (equivalent to something in $S$, up to global phase); if it's diagonal but *not* a multiple of the identity, it acts like a logical $\bar{Z}$; if it's off-diagonal, it swaps $|0_L\rangle \leftrightarrow |1_L\rangle$, like a logical $\bar{X}$ (or $\bar{Y}$ — we don't separate those two here; see notebook 07 for the fully general treatment).

In [5]:
def logical_action_matrix(pauli_string, state_0L, state_1L):
    P = pauli_string_matrix(pauli_string)
    return np.array([
        [np.vdot(state_0L, P @ state_0L), np.vdot(state_0L, P @ state_1L)],
        [np.vdot(state_1L, P @ state_0L), np.vdot(state_1L, P @ state_1L)],
    ])


def classify(pauli_string, state_0L, state_1L):
    L = logical_action_matrix(pauli_string, state_0L, state_1L)
    off_diagonal_zero = np.isclose(abs(L[0, 1]), 0) and np.isclose(abs(L[1, 0]), 0)
    if off_diagonal_zero and np.isclose(L[0, 0], L[1, 1]):
        return "trivial (in S, up to phase)"
    if off_diagonal_zero:
        return "logical Z-like (diagonal, unequal)"
    return "logical X-like (off-diagonal)"


for p in sorted(centralizer, key=weight):
    print(f"{p}  weight={weight(p)}  {classify(p, repetition_0L, repetition_1L)}")

III  weight=0  trivial (in S, up to phase)
IIZ  weight=1  logical Z-like (diagonal, unequal)
IZI  weight=1  logical Z-like (diagonal, unequal)
ZII  weight=1  logical Z-like (diagonal, unequal)
IZZ  weight=2  trivial (in S, up to phase)
ZIZ  weight=2  trivial (in S, up to phase)
ZZI  weight=2  trivial (in S, up to phase)
XXX  weight=3  logical X-like (off-diagonal)
XXY  weight=3  logical X-like (off-diagonal)
XYX  weight=3  logical X-like (off-diagonal)
XYY  weight=3  logical X-like (off-diagonal)
YXX  weight=3  logical X-like (off-diagonal)
YXY  weight=3  logical X-like (off-diagonal)
YYX  weight=3  logical X-like (off-diagonal)
YYY  weight=3  logical X-like (off-diagonal)
ZZZ  weight=3  logical Z-like (diagonal, unequal)


Four elements are trivial — exactly $S = \{III, ZZI, IZZ, ZIZ\}$, the stabilizer group itself (size $2^m = 4$). Everything else in the centralizer is a genuine logical operator. Notice `ZII`, `IZI`, and `IIZ` all classify as logical-$Z$-like at weight 1: any single physical $Z$ already distinguishes $|000\rangle$ from $|111\rangle$, because this code was only ever built to protect against **bit flips** — it has no protection against phase flips at all. This is worth stating plainly: a code's distance is not always one single number for every kind of error; here, the *bit-flip* distance is 3 (matching notebook 02) but the *phase-flip* distance is only 1. Section 6 accounts for this by searching for $\bar{X}$-like and $\bar{Z}$-like operators separately.

## 5. Logical equivalence: different operators, identical action

`XXX`, `XXI`... wait — look for two *different* weight-3-or-less logical-$X$-like operators in the list above and check whether they act identically on the code space (equivalently, whether they differ by a stabilizer element).

In [6]:
x_like = [p for p in centralizer if classify(p, repetition_0L, repetition_1L) == "logical X-like (off-diagonal)"]
print("Logical-X-like elements:", sorted(x_like, key=weight))

p1, p2 = "XXX", "YYX"  # differ by (XXX)(YYX)^-1 -- check if that product lies in S
def multiply_pauli_strings(p, q):
    # single-qubit Pauli multiplication table, tracking only which letter results (ignoring global phase)
    table = {
        ("I", c): c for c in "IXYZ"
    }
    table.update({(c, "I"): c for c in "IXYZ"})
    table.update({(c, c): "I" for c in "XYZ"})
    table.update({("X", "Y"): "Z", ("Y", "X"): "Z", ("Y", "Z"): "X", ("Z", "Y"): "X", ("Z", "X"): "Y", ("X", "Z"): "Y"})
    return "".join(table[(a, b)] for a, b in zip(p, q))


product = multiply_pauli_strings(p1, p2)
print(f"{p1} times {p2} (ignoring global phase) = {product}")
print(f"Is that product in the stabilizer group S = {repetition_generators + ['III']} (up to relabeling)?")
print("  i.e. does it act trivially?", classify(product, repetition_0L, repetition_1L) == "trivial (in S, up to phase)")

L1 = logical_action_matrix(p1, repetition_0L, repetition_1L)
L2 = logical_action_matrix(p2, repetition_0L, repetition_1L)
print(f"\nLogical action of {p1}:\n{np.round(L1, 3)}")
print(f"Logical action of {p2}:\n{np.round(L2, 3)}")
print("Same action on the code space (up to the global phase from Y's i's):", np.allclose(np.abs(L1), np.abs(L2)))

Logical-X-like elements: ['XXX', 'XXY', 'XYX', 'XYY', 'YXX', 'YXY', 'YYX', 'YYY']
XXX times YYX (ignoring global phase) = ZZI
Is that product in the stabilizer group S = ['ZZI', 'IZZ', 'III'] (up to relabeling)?
  i.e. does it act trivially? True

Logical action of XXX:
[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]
Logical action of YYX:
[[ 0.+0.j -1.+0.j]
 [-1.+0.j  0.+0.j]]
Same action on the code space (up to the global phase from Y's i's): True


`XXX` and `YYX` differ by a stabilizer element (their product, ignoring an overall phase, lands back in $S$), and — up to that phase — they act identically on the code space. This is **logical equivalence**: any two operators in the same coset of $S$ within $C(S)$ are interchangeable as far as the encoded logical information is concerned. A decoder is always free to apply *any* representative of the correct coset, exactly like Section 4 of notebook 05 chose an arbitrary qubit to correct within a degenerate block.

## 6. Code distance, computed one way for every code

**Definition:** the distance of a stabilizer code is the minimum weight of an operator in $C(S) \setminus S$ — a nontrivial logical operator. Because a code can protect $X$-type and $Z$-type errors differently (Section 4's repetition-code example), it's more informative to search for the minimum-weight $\bar{X}$-like and $\bar{Z}$-like operators separately, then report both.

In [7]:
def all_weight_le(max_weight, n):
    strings = []
    for w in range(1, max_weight + 1):
        for positions in combinations(range(n), w):
            for letters in iproduct("XYZ", repeat=w):
                s = ["I"] * n
                for pos, letter in zip(positions, letters):
                    s[pos] = letter
                strings.append("".join(s))
    return strings


def code_distance(generator_strings, state_0L, state_1L, n_qubits, max_weight):
    x_like_min, z_like_min = None, None
    for w in range(1, max_weight + 1):
        for s in all_weight_le(w, n_qubits):
            if weight(s) != w:
                continue
            if not all(commutes(s, g) for g in generator_strings):
                continue
            kind = classify(s, state_0L, state_1L)
            if kind == "logical X-like (off-diagonal)" and x_like_min is None:
                x_like_min = w
            elif kind == "logical Z-like (diagonal, unequal)" and z_like_min is None:
                z_like_min = w
        if x_like_min is not None and z_like_min is not None:
            break
    return x_like_min, z_like_min


rep_dx, rep_dz = code_distance(repetition_generators, repetition_0L, repetition_1L, n_qubits=3, max_weight=3)
print(f"Repetition code: X-distance={rep_dx}, Z-distance={rep_dz}")

shor_dx, shor_dz = code_distance(shor_generators, shor_0L, shor_1L, n_qubits=9, max_weight=3)
print(f"Shor code:       X-distance={shor_dx}, Z-distance={shor_dz}")

Repetition code: X-distance=3, Z-distance=1
Shor code:       X-distance=3, Z-distance=3


### Prediction 2

**Before running the next cell:** the Steane code is a CSS code built symmetrically from the same $H$ for both $X$-type and $Z$-type checks (notebook 05, Section 7). Do you expect its $X$-distance and $Z$-distance to be equal or different? Compare that expectation to the repetition code's asymmetric result above.

In [8]:
steane_0L_and_1L_source = projector_onto_plus_one(steane_generators, n_qubits=7)
# Recover two orthonormal code-space basis vectors from the projector's column space.
eigvals, eigvecs = np.linalg.eigh(steane_0L_and_1L_source)
code_space_basis = eigvecs[:, np.isclose(eigvals, 1)]
steane_0L = code_space_basis[:, 0]
steane_1L = code_space_basis[:, 1]

steane_dx, steane_dz = code_distance(steane_generators, steane_0L, steane_1L, n_qubits=7, max_weight=3)
print(f"Steane code:     X-distance={steane_dx}, Z-distance={steane_dz}")

Steane code:     X-distance=3, Z-distance=3


As expected from its symmetric CSS construction, the Steane code protects $X$ and $Z$ errors equally: $d_X = d_Z = 3$, matching notebook 05's direct construction from the self-dual-containing Hamming code. The repetition code, by contrast, is lopsided ($d_X = 3$, $d_Z = 1$) precisely because it was only ever built with $Z$-type checks watching for $X$ errors. Shor's code, built by nesting two repetition codes with opposite roles (Section 2 of notebook 05), ends up symmetric at $d_X = d_Z = 3$ even though each individual layer is as lopsided as the plain repetition code — nesting a bit-flip-only code inside a phase-flip-only code is what restores the missing protection.

### Prediction 3

**Before moving on:** given that the repetition code from notebooks 02–03 has $d_Z = 1$, what would happen if you fed it into notebook 03's syndrome machinery and injected a single $Z$ error instead of an $X$ error? Would either of the $Z_0Z_1$/$Z_1Z_2$ stabilizers fire?

(Answer, briefly: no. $Z$ always commutes with $Z$, so a lone $Z$ error is invisible to $Z$-type checks — consistent with $d_Z = 1$: a single physical $Z$ operator on any one data qubit is already a full-strength, undetectable logical operator for this code.)

## Exercises

**1.** Write `find_all_minimum_weight_logicals(generator_strings, state_0L, state_1L, n_qubits, distance)` that, given a code's already-known distance, returns *every* Pauli string of exactly that weight classified as `"logical X-like"` (reuse `code_distance`'s scanning loop, but don't stop at the first hit). Run it on the repetition code at weight 3 — how many minimum-weight logical-$\bar{X}$ representatives are there, and are they all logically equivalent to each other (Section 5)?

**2.** Section 4 showed the repetition code's stabilizer group $S$ has exactly $2^m = 4$ elements out of the $64$ in the full 3-qubit Pauli group, and its centralizer $C(S)$ has 16. Write `count_cosets(centralizer_size, stabilizer_size)` that returns how many cosets of $S$ partition $C(S)$, and relate that number to $k$ (Section 2) — for a code with $k$ logical qubits, how many cosets should there always be, and why (hint: each coset corresponds to one Pauli acting on the $k$-qubit logical space — how many single/multi-qubit Pauli operators act on $k$ logical qubits)?

In [9]:
def find_all_minimum_weight_logicals(generator_strings, state_0L, state_1L, n_qubits, distance):
    pass  # TODO: implement, then remove this stub body


def count_cosets(centralizer_size, stabilizer_size):
    pass  # TODO: implement, then remove this stub body


# TODO: once implemented, try e.g.:
# print(find_all_minimum_weight_logicals(repetition_generators, repetition_0L, repetition_1L, 3, rep_dx))
# print(count_cosets(len(centralizer), len(repetition_generators + ["III"])))